# Modeling and Evaluation Plan

This notebook describes the modeling workflow and evaluation strategy for predicting tornado occurrences from station-level meteorological data.  
We implement and evaluate two models:

1. **Decision Tree Classifier** — interpretable baseline.
2. **XGBoost Classifier** — robust, high-performance ensemble.

The evaluation framework explicitly addresses data leakage, imbalance, and generalization, in alignment with the **Checkpoint 3: Model Evaluation Plan** guidelines.

---

## **1. Unit of Analysis**

Each row in the dataset represents a **station-hour observation** that merges:
- Meteorological readings from the nearest weather station.
- Tornado occurrence metadata (begin/end latitudes, longitudes, time).

Our prediction target is a Boolean variable:
> `TORNADO_OCCURRENCE ∈ {0, 1}` — whether a tornado was active near the station within the prior `time_window` hours.

---

## **2. Data Splitting and Leakage Mitigation**

We guard against multiple types of data leakage:

| Leakage Type | Description | Mitigation Strategy |
|---------------|--------------|---------------------|
| **Temporal leakage** | Future data influencing past predictions. | Split data chronologically (training on earlier timestamps). |
| **Geographic leakage** | Nearby stations sharing correlated meteorological conditions. | Group by spatial cluster or station ID for cross-validation (use `GroupKFold` or `GroupShuffleSplit`). |
| **Feature leakage** | Features derived directly from tornado labels. | Exclude latitude/longitude and derived “distance” columns from modeling. |
| **Overfitting the split** | Tweaking models to one partition. | Use multiple CV folds, reserve a *final untouched test set*. |




| Metric                | Purpose                                                      |
| --------------------- | ------------------------------------------------------------ |
| **Precision**         | Avoid false positives (predicting tornado when none occurs). |
| **Recall**            | Avoid false negatives (missing tornado events).              |
| **F1-score**          | Balanced tradeoff for imbalanced classes.                    |
| **ROC-AUC / PR-AUC**  | Threshold-independent model quality.                         |
| **Calibration curve** | Checks probabilistic reliability.                            |
